In [1]:
import nltk
nltk.download('punkt')
nltk.download('vader_lexicon')
nltk.download("punkt_tab")


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\barna\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\barna\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\barna\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

# Data Labeling with VADER and LLM (Chatgpt-4o-mini)

In [ ]:
import pandas as pd
from tqdm import tqdm
from nltk.sentiment import SentimentIntensityAnalyzer
from nltk import sent_tokenize
from openai import OpenAI
from dotenv import load_dotenv
import os

# setup
sia = SentimentIntensityAnalyzer()

def vader_label(text_list):
    text = " ".join(text_list)
    sentences = sent_tokenize(text)
    scores = [sia.polarity_scores(s)['compound'] for s in sentences]
    if not scores:
        return 'neutral'
    avg = sum(scores) / len(scores)
    if avg > 0.05:
        return 'positive'
    elif avg < -0.05:
        return 'negative'
    else:
        return 'neutral'
    
# OpenAI Setup
load_dotenv()
OPENAI_API_KEY = os.getenv("API_KEY")

client = OpenAI(
    api_key=OPENAI_API_KEY
)

def llm_label(text_list, model="gpt-4o-mini"):
    text = " ".join(text_list)
    prompt = f"""
You are an NLP annotator. Classify the overall sentiment of this news article toward video-based social media
(e.g., YouTube, Instagram, Snapchat) and youth mental health as one of: "positive", "neutral", or "negative".
Respond with only one word.

Article:
{text[:3000]}  # truncates long text
"""
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            temperature=0
        )
        output = response.choices[0].message.content.strip().lower()
        # sanitize model response
        if "pos" in output:
            return "positive"
        elif "neg" in output:
            return "negative"
        elif "neu" in output:
            return "neutral"
        return "neutral"
    except Exception as e:
        print("Error:", e)
        return "neutral"

In [3]:
# load and label dataset
file_path = './news-snapchat-noagegroup-mental-2024.jsonl'
data = pd.read_json(file_path, lines=True)

vader_labels = []
llm_labels = []

for _, row in tqdm(data.iterrows(), total=len(data)):
    text = row['body']
    vader_labels.append(vader_label(text))
    llm_labels.append(llm_label(text))

data['vader_sentiment'] = vader_labels
data['llm_sentiment'] = llm_labels

100%|██████████| 676/676 [05:02<00:00,  2.23it/s]


In [ ]:
# Save the results
output_file = "labeled_snapchat_comparison.jsonl"
data[['title', 'body', 'provider', 'publication_date', 'vader_sentiment', 'llm_sentiment']].to_json(
    output_file, orient='records', lines=True
)

print(f"Saved labeled file to {output_file}")

# Quick comparison summary between Vadar and LLM sentiment categorization
print("VADER vs LLM label comparison:")
print(data[['vader_sentiment', 'llm_sentiment']].value_counts())
agreement = (data['vader_sentiment'] == data['llm_sentiment']).mean()
print(f"Label agreement: {agreement:.2%}")


Saved labeled file to labeled_snapchat_comparison.jsonl

VADER vs LLM label comparison:
vader_sentiment  llm_sentiment
neutral          negative         363
negative         negative         154
positive         negative         112
                 neutral           22
                 positive          12
neutral          neutral            8
negative         neutral            4
neutral          positive           1
Name: count, dtype: int64

Label agreement: 25.74%


# Further Data Analysis
1. Comparisons between labels between VADER and LLM
2. Word counts, sentiments, and platform comparisons in the deduplicated dataset

In [4]:
import pandas as pd

# load dataset files
snap = pd.read_json(r"labeled_snapchat_comparison.jsonl", lines=True)
yt   = pd.read_json(r"labeled_youtube_comparison.jsonl",   lines=True)

# combine datasets
data = pd.concat([snap, yt], ignore_index=True)

# load full confusion matrix
cm = data.groupby(["llm_sentiment", "vader_sentiment"]).size().unstack(fill_value=0)
print("\nFull Confusion Matrix (LLM = ground truth):")
print(cm)

# show alignment rate as an average
agreement = (data["llm_sentiment"] == data["vader_sentiment"]).mean()
print(f"\nLabel Agreement: {agreement:.2%}")



Full Confusion Matrix (LLM = ground truth):
vader_sentiment  negative  neutral  positive
llm_sentiment                               
negative              556      573       235
neutral                39      112       321
positive                1       12       153

Label Agreement: 41.01%


In [ ]:
import pandas as pd
import numpy as np

# doing further analysis. Loading datasets
snap = pd.read_json(r"labeled_snapchat_comparison.jsonl", lines=True)
yt   = pd.read_json(r"labeled_youtube_comparison.jsonl",   lines=True)

snap["platform"] = "snapchat"
yt["platform"]   = "youtube"

data = pd.concat([snap, yt], ignore_index=True)

print("Loaded Snapchat:", len(snap))
print("Loaded YouTube:", len(yt))
print("Combined:", len(data))

# word count
def word_count(body):
    if isinstance(body, list):
        body = " ".join(body)
    return len(body.split())

data["word_count"] = data["body"].apply(word_count)

# normalize text for dedup
def normalize_text(x):
    if isinstance(x, list):
        x = " ".join(x)
    return str(x).strip().lower()

data["title_norm"] = data["title"].str.strip().str.lower()
data["body_norm"]  = data["body"].apply(normalize_text)


dedup = data.drop_duplicates(subset=["title_norm", "body_norm"])

print("Deduplication Stats")
print("Original total:", len(data))
print("Deduped total:", len(dedup))
print("Removed duplicates:", len(data) - len(dedup))

# print overall sentiments
print(dedup["llm_sentiment"].value_counts())
print("Proportions:")
print(dedup["llm_sentiment"].value_counts(normalize=True))

# platform specific sentiment
print("Snapchat sentiment")
snap_d = dedup[dedup["platform"] == "snapchat"]
print(snap_d["llm_sentiment"].value_counts())
print("Proportions:")
print(snap_d["llm_sentiment"].value_counts(normalize=True))

print("YouTube sentiment")
yt_d = dedup[dedup["platform"] == "youtube"]
print(yt_d["llm_sentiment"].value_counts())
print("Proportions:")
print(yt_d["llm_sentiment"].value_counts(normalize=True))

# word count stats
print("Average word count")
print(dedup["word_count"].describe())

print("Snapchat word count:")
print(snap_d["word_count"].describe())

print("YouTube word count:")
print(yt_d["word_count"].describe())

# save deduped dataset
output_jsonl = "deduped_news_dataset.jsonl"
dedup.to_json(output_jsonl, orient="records", lines=True, force_ascii=False)

print("Rows saved:", len(dedup))

# per-platform datsets
snap_jsonl = "deduped_snapchat.jsonl"
yt_jsonl   = "deduped_youtube.jsonl"

snap_d.to_json(snap_jsonl, orient="records", lines=True, force_ascii=False)
yt_d.to_json(yt_jsonl, orient="records", lines=True, force_ascii=False)

print("Saved platform subsets:")
print(" -", snap_jsonl, "(", len(snap_d), "rows )")
print(" -", yt_jsonl,   "(", len(yt_d),   "rows )")


Loaded Snapchat: 676
Loaded YouTube: 1326
Combined: 2002
Deduplication Stats
Original total: 2002
Deduped total: 1508
Removed duplicates: 494
llm_sentiment
negative    983
neutral     368
positive    157
Name: count, dtype: int64
Proportions:
llm_sentiment
negative    0.651857
neutral     0.244032
positive    0.104111
Name: proportion, dtype: float64
Snapchat sentiment
llm_sentiment
negative    546
neutral      33
positive     11
Name: count, dtype: int64
Proportions:
llm_sentiment
negative    0.925424
neutral     0.055932
positive    0.018644
Name: proportion, dtype: float64
YouTube sentiment
llm_sentiment
negative    437
neutral     335
positive    146
Name: count, dtype: int64
Proportions:
llm_sentiment
negative    0.476035
neutral     0.364924
positive    0.159041
Name: proportion, dtype: float64
Average word count
count    1508.000000
mean      591.233422
std       473.215753
min        14.000000
25%       333.000000
50%       512.000000
75%       739.500000
max      6808.000000
N

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# plotting and saving data as graphs
overall_counts = dedup["llm_sentiment"].value_counts().reindex(["negative", "neutral", "positive"])

plt.figure(figsize=(6,4))
overall_counts.plot(kind="bar", color=["#d62728", "#ffbf00", "#2ca02c"])
plt.title("Overall Sentiment Distribution (Deduped)")
plt.xlabel("Sentiment")
plt.ylabel("Count")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("overall_sentiment.png", dpi=300)
plt.close()


# platform specific sentiment distribution
platform_table = (
    dedup.groupby(["platform", "llm_sentiment"])
         .size()
         .unstack(fill_value=0)
         .reindex(columns=["negative", "neutral", "positive"])
)

plt.figure(figsize=(7,5))
platform_table.plot(kind="bar", color=["#d62728", "#ffbf00", "#2ca02c"])
plt.title("Sentiment Distribution by Platform (Deduped)")
plt.xlabel("Platform")
plt.ylabel("Count")
plt.xticks(rotation=0)
plt.legend(title="Sentiment")
plt.tight_layout()
plt.savefig("platform_sentiment.png", dpi=300)
plt.close()


# word count histogram
plt.figure(figsize=(7,5))
plt.hist(dedup["word_count"], bins=40, color="#1f77b4", edgecolor="black", alpha=0.8)
plt.title("Distribution of Article Word Counts")
plt.xlabel("Word Count")
plt.ylabel("Frequency")
plt.tight_layout()
plt.savefig("wordcount_hist.png", dpi=300)
plt.close()

print("Saved figures: overall_sentiment.png, platform_sentiment.png, wordcount_hist.png")


Saved figures: overall_sentiment.png, platform_sentiment.png, wordcount_hist.png


<Figure size 700x500 with 0 Axes>